In [4]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "none"#"last_expr_or_assign"

import numpy as np
import matplotlib.pyplot as plt
from util.logger import EventTracker
from util.basis_scaled import *
from util.plot_tools import *
from boundary_solvers.gauss_grid_2d import StokesDirichletProblem
from scipy.io import loadmat
import matplotlib.pyplot as plt
from util.interp import PiecewiseInterp2D
from boundary_solvers.gauss_grid_2d import TrapezGrid
from hmm.stokes import *
from hmm.stokes_deep import DeepMicroSolver, get_net
import matplotlib
from scipy.interpolate import interp1d
from architecture.session import fno_ver3, fno_ver4, fno_ver1, fno_ver2, egeofno_ver2, svdfno_ver2, svdfno_ver1, svdfno_ver3, egeofno_ver5, svdfno_ver4
import torch
from boundary_solvers.geometry import GPDomain
#matplotlib.rcParams['text.usetex'] = False


#net_dir = "/mnt/data0/emastr/article_training_nodecay/"
net_dir = "/mnt/data0/emastr/training/article_training_hugedata/"
MESH_PATH = "/home/emastr/deep-micro-slip-model/data/mesh/"
figures_dir = "/home/emastr/deep-micro-slip-model/data/figures/"
simulation_dir = "/home/emastr/deep-micro-slip-model/data/stokes_fenics/"
#data_dir = "/home/emastr/deep-micro-slip-model/data/micro_geometries_boundcurv_repar_256_torch/data_big_clean.torch"
run_dir = "/home/emastr/deep-micro-slip-model/data/reference_2/"

#pretty_pyplot_layout()

In [5]:
device = "cpu" # "cuda:0"
num_pts = 256
#net, net_settings = get_net(f"{net_dir}fnoskip_big_data_{40000}.Torch", 256, "cpu", "float")
net_type = 2# 0: egeofno, 1: svdfno, 2: fno_vanilla
# net_type 0 works well

if net_type == 0:
    net = egeofno_ver1(device=device)
    net_data = torch.load(f"{net_dir}fno_ver1_seed0_40000.Torch", map_location=torch.device(device))
elif net_type == 1:
    net = svdfno_ver2(device=device)
    net_data = torch.load(f"{net_dir}svdfno_ver2_seed0_40000.Torch", map_location=torch.device(device))
else:
    net = fno_ver3(device=device)
    net_data = torch.load(f"{net_dir}fno_vanilla_ver3_seed0_40000.Torch", map_location=torch.device(device))

net.load_state_dict(net_data["state dict"])
net_settings =  {"num_pts": num_pts, \
                 "input_features": net_data["settings"]["input_features"], \
                 "output_features": net_data["settings"]["output_features"],\
                 "device": device, "dtype": torch.float}
#print(net_settings["input_features"])
print(net_data["settings"]["output_features"])

['rx', 'ry', 'drx_norm', 'dry_norm']


In [ ]:
logger = EventTracker() # Log time


eps = 1.0
nMic = 13#7 #freq_g*2+1#2*(round(Lx/(30*eps)) // 2) + 1 # 15 # 15 #nMic*2 +1
width = 8*eps #5*eps #3
height = width*0.8 #3
n_refine = 0
linePos = 0.03


    
class StokesMicProb(MicroProblem):
    def __init__(self, width, height, linePos, deg_project=None, xDim_reduce=None, yDim_reduce=None, logger=None, **kwargs):
        self.width = width
        self.height = height
        self.linePos = linePos
        self.deg_project = deg_project
        self.logger = logger
        self.xDim_reduce = xDim_reduce
        self.yDim_reduce = yDim_reduce
        
        
        self.geom = GPDomain("exp", 
                    shape=.05, 
                    num=20, 
                    scale=.05, 
                    bound=.3, 
                    width=1, 
                    height=1, 
                    corner_w = 0.3,
                    line_pos= 0.2, 
                    n_refine=2,#1, 
                    n_corner_refine=0)  
        self.condition = None
        
    
    def is_solution(self, micro_sol, tol = 1e-5):
        # Requires a solver to check. instead, check convergence.
        return False
    
    def update(self, macro_sol, **kwargs):
        """Given a solution macroSol to the macro problem, read off the solution at points in the micro problem,
        and extrapolate to missing parts of the micro boundary."""
        
        # Joint components
        u = macro_sol.u
        v = macro_sol.v

        if (self.yDim_reduce is None) or (self.xDim_reduce is None):
           yDim = u.yDim
           xDim = u.xDim
        else:
           yDim = self.yDim_reduce
           xDim = self.xDim_reduce
        u_ = u.change_dim(xDim, yDim)
        v_ = v.change_dim(xDim, yDim)
        
        def toComplex(fx, fy):
            return lambda z: fx(np.real(z), np.imag(z)) + 1j * fy(np.real(z), np.imag(z))
        
        if self.logger is not None:
            self.logger.start_event("micro_update_diff")

        U = toComplex(u_, v_)
        dxU = toComplex(u_.diff(1,0), v_.diff(1,0))
        dyU = toComplex(u_.diff(0,1), v_.diff(0,1))
        dxdxU = toComplex(u_.diff(2,0), v_.diff(2,0))
        dxdyU = toComplex(u_.diff(1,1), v_.diff(1,1))
        dydyU = toComplex(u_.diff(0,2), v_.diff(0,2))
        
        if self.logger is not None:
            self.logger.end_event("micro_update_diff")

        z  = lambda t: self.geom.eval_param(t=t)
        dz = lambda t: self.geom.eval_param(t=t, derivative=1)
        ddz = lambda t: self.geom.eval_param(t=t, derivative=2)

        def g(t):
            return U(z(t))

        def dg(t):
            z_ = z(t)
            dz_ = dz(t)
            return np.real(dz_) * dxU(z_) + np.imag(dz_) * dyU(z_)

        def ddg(t):
            z_ = z(t)
            dz_ = dz(t)
            ddz_ = ddz(t)

            x, y = np.real(z_), np.imag(z_)
            dx, dy = np.real(dz_), np.imag(dz_)
            ddx, ddy = np.real(ddz_), np.imag(ddz_)    

            return ddx * dxU(z_) + ddy * dyU(z_) + (dx**2) * dxdxU(z_) + (2*dx*dy)*dxdyU(z_) + (dy**2)*dydyU(z_)
        
        if self.logger is not None:
            self.logger.start_event("micro_update_fit")

        deg_project = kwargs.pop("N", self.deg_project)
        g_extrap = self.geom.project(g, dg, ddg, N=deg_project, **kwargs)
        self.set_data(g_extrap)

        if self.logger is not None:
            self.logger.end_event("micro_update_fit")
    
    def set_data(self, condition):
        """Change the boundary condition of the micro problem."""
        self.condition = condition
    
    def plot(self, axis, **kwargs):
        """Plot micro domain. relative = True means plotting in true coordinates."""
        self.geom.plot(axis, **kwargs)
        
micros = [StokesMicProb(width, height, linePos, deg_project=8, logger=logger, n_refine=n_refine, xDim_reduce=xDim_reduce, yDim_reduce=yDim_reduce) for x in xPos]



deep_micro_solvers = [DeepMicroSolver(m, net, net_settings, logger=logger, tol=tol) for m in micros]
micro_solvers = [MicroSolver(m, logger=logger, tol=tol) for m in micros]


